In [5]:

import socket
import os
import gradio as gr
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ==========================================
# 0. 서버 환경 설정 및 유틸리티 함수
# ==========================================
GRADIO_HOST = "127.0.0.1"
GRADIO_PORT = 7860
OLLAMA_BASE_URL = "http://192.168.45.175:11434"
DATA_DIR = "../data/data"

def close_demo() -> None:
    gr.close_all()

def launch_demo(demo: gr.Blocks, port: int = GRADIO_PORT) -> None:
    close_demo()
    demo.launch(server_name=GRADIO_HOST, server_port=port, prevent_thread_lock=True)


def get_llm_ollama(
    model: str = "qwen3:8b-q4_K_M",
    temperature: float = 0.5,
    seed: int | None = None,
    verbose: bool = True,
    **kwargs
) -> ChatOllama:
    """Ollama 기반의 ChatOllama 인스턴스를 생성합니다.
    verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOllama에 전달할 추가 옵션 (예: base_url, timeout 등)"""
    
    llm = ChatOllama(
        model=model,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성 향상
        base_url=OLLAMA_BASE_URL, # 상수를 base_url 인자에 명시적으로 전달
        **kwargs,                  # **kwargs는 항상 맨 마지막에 위치해야 합니다.
    )
    
    if verbose:
        print(f"[LLM 생성] model={llm.model} | base_url={OLLAMA_BASE_URL} | temperature={temperature}")
        
    return llm


In [7]:

# ==========================================
# 1. Ollama RAG 파이프라인 구축
# ==========================================
print("1. Ollama 임베딩 및 LLM 초기화 중...")
embeddings = OllamaEmbeddings(
    model="nomic-embed-text", ##### 중요 로컬에서 설치되어 있어야 하는 embeding 모델
    base_url=OLLAMA_BASE_URL,
)

llm = get_llm_ollama(model="qwen3:8b-q4_K_M", temperature=0.5) ##### Ollama 에서 돌리는 모델

print(f"2. '{DATA_DIR}' 폴더에서 PDF 문서 로드 및 벡터스토어 구축 중...")
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)

loader = DirectoryLoader(DATA_DIR, glob="*.pdf", loader_cls=PyPDFLoader)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=300)
split_docs = text_splitter.split_documents(docs)

vectorstore = FAISS.from_documents(split_docs, embeddings)


# 상위 가지고 올 수
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})



prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 문서 내용을 바탕으로 질문에 답하세요. 문서에 없는 내용은 모른다고 답하세요.\n\n[문서]\n{context}"),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_sources = (
    RunnableParallel(docs=retriever, question=RunnablePassthrough())
    .assign(answer=answer_chain)
)
print("✅ Ollama RAG 체인 준비 완료!")


# ==========================================
# 2. 챗봇 및 출처 연동 로직 (전체 목록 아코디언 포맷 적용)
# ==========================================
ERROR_PREFIX = "⚠️ 답변 생성 중 오류가 발생했습니다: "

def format_sources_markdown(docs) -> str:
    """좌측 패널에 전체 참고 목록을 접어두고 클릭 시 열리도록 생성"""
    if not docs:
        return "아직 검색된 참고 문서가 없습니다."
    
    # 개별 항목 리스트 생성
    items_html = ""
    for i, doc in enumerate(docs):
        source_name = os.path.basename(doc.metadata.get("source", "Unknown"))
        page_num = doc.metadata.get("page", "N/A")
        content_preview = doc.page_content.strip().replace("\n", " ")
        
        items_html += f"""
<details style="margin-bottom: 6px; border: 1px solid #444; border-radius: 4px; overflow: hidden;">
<summary style="padding: 6px 10px; background-color: #2d2d2d; color: #ffffff; cursor: pointer; font-size: 0.9em; font-weight: bold;">
    📂 [{i + 1}] {source_name} &nbsp;|&nbsp; <span style="color: #4daafc;">Page {page_num}</span>
</summary>
<div style="padding: 10px; background-color: #1e1e1e; color: #d4d4d4; font-size: 0.85em; border-top: 1px solid #444; line-height: 1.4;">
    {content_preview}
</div>
</details>
"""

    # 전체 목록 자체를 감싸는 상위 대형 아코디언 (<details>) 적용
    output = f"""
<details open style="border: 1px solid #007bff; border-radius: 6px; overflow: hidden; background-color: #252526;">
<summary style="padding: 10px 14px; background-color: #007bff; color: #ffffff; cursor: pointer; font-weight: bold; font-size: 0.95em;">
    📄 참고된 PDF 문서 목록 (클릭하여 접기/펼치기)
</summary>
<div style="padding: 12px; background-color: #1e1e1e;">
    <div style="margin-bottom: 10px; color: #9cdcfe; font-size: 0.85em;">📁 경로: <code>{DATA_DIR}/</code></div>
    {items_html}
</div>
</details>
"""
    return output

def chat_respond(message, chat_history):
    """질문을 받아 스트리밍 답변을 생성하고, 좌측 패널과 우측 패널을 동시에 업데이트한다."""
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": ""})
    
    try:
        retrieved_result = rag_chain_with_sources.invoke(message)
        docs = retrieved_result["docs"]
        
        sources_md = format_sources_markdown(docs)
        
        partial_message = ""
        for chunk in answer_chain.stream({"docs": docs, "question": message}):
            partial_message += chunk
            chat_history[-1]["content"] = partial_message
            yield "", chat_history, sources_md
            
    except Exception as e:
        chat_history[-1]["content"] = f"{ERROR_PREFIX}{e}"
        yield "", chat_history, "오류가 발생했습니다."


# ==========================================
# 3. 3:7 비율 좌우 분할 Gradio UI 생성 및 실행
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 로컬 Ollama RAG PDF 챗봇 (3:7 레이아웃)")
    
    with gr.Row():
        
        # [왼쪽 패널: 3 비율] - 접이식 상위 아코디언 출처 목록 영역
        with gr.Column(scale=3):
            gr.Markdown("### 📂 참조 PDF 및 연관 내용")
            sources_markdown = gr.Markdown(
                value="질문을 입력하시면 참고된 문서 목록이 생성되며, 클릭하여 내용을 확인하실 수 있습니다.",
                elem_id="sources_panel"
            )
        
        # [오른쪽 패널: 7 비율] - 메인 대화창
        with gr.Column(scale=7):
            gr.Markdown("### 💬 메인 대화")
            chatbot = gr.Chatbot(label="대화창", height=480)
            msg = gr.Textbox(label="질문을 입력하세요 (Enter)", placeholder="PDF 문서 내용에 대해 무엇이든 물어보세요...")
            
            with gr.Row():
                clear = gr.Button("대화 초기화")

    msg.submit(chat_respond, [msg, chatbot], [msg, chatbot, sources_markdown])
    clear.click(lambda: ([], "아직 검색된 참고 문서가 없습니다."), None, [chatbot, sources_markdown], queue=False)

# 포트 충돌 방지 실행
launch_demo(demo, port=7861)

1. Ollama 임베딩 및 LLM 초기화 중...
[LLM 생성] model=qwen3:8b-q4_K_M | base_url=http://192.168.45.175:11434 | temperature=0.5
2. '../data/data' 폴더에서 PDF 문서 로드 및 벡터스토어 구축 중...
✅ Ollama RAG 체인 준비 완료!


C:\Users\user\AppData\Local\Temp\ipykernel_15276\2263586748.py:119: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Users\user\Desktop\Langchain\biztalk-antigravity\FastApi\.venv\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\user\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다
c:\Users\user\Desktop\Langchain\biztalk-antigravity\FastApi\.venv\Lib\site-packages\gradio\routes.

In [3]:
def is_port_open(port: int = GRADIO_PORT, host: str = GRADIO_HOST) -> bool:
    """해당 포트에서 서버가 응답하면 True(점유 중), 아니면 False(해제됨)를 반환한다."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)                       # 응답이 없으면 0.5초 뒤 포기
        return s.connect_ex((host, port)) == 0  # 접속 성공(0)이면 누군가 포트를 쓰는 중


close_demo()
print("포트 점유 여부:", is_port_open())   # False면 해제 완료

포트 점유 여부: False
